# LSTM-MI — Binary benign/DGA classifier (bản đã sửa logic)

Notebook này giữ đúng mục tiêu **0 = benign, 1 = DGA**. Các sửa đổi chính:

- dùng `pack_padded_sequence`, nên LSTM lấy hidden state ở ký tự cuối thật của domain thay vì cuối vùng padding;
- tính class weights đúng chiều: lớp DGA thiểu số nhận trọng số lớn hơn;
- loại domain trùng/nhãn mâu thuẫn trong từng tập và loại overlap theo thứ tự train → validation → test;
- đo thời gian ngay trong train loop, không train thêm lần hai bằng optimizer của model cũ;
- test chỉ được dùng sau khi đã chọn checkpoint bằng validation F1;
- checkpoint chỉ chứa tensor và kiểu dữ liệu cơ bản, nạp bằng `weights_only=True` trên PyTorch mới;
- train, test và dự đoán một domain dùng cùng một hàm làm sạch/encode.

File gốc `LSTM_MI.ipynb` không bị thay đổi.


In [1]:
# Colab: mount Drive nếu đang chạy trên Colab.
# Đổi DRIVE_PROJECT_DIR nếu thư mục của bạn có tên/đường dẫn khác.
from pathlib import Path
import os

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/LSTM.MI')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    if not DRIVE_PROJECT_DIR.is_dir():
        raise FileNotFoundError(f'Không tìm thấy: {DRIVE_PROJECT_DIR}')
    os.chdir(DRIVE_PROJECT_DIR)
    print('Working directory:', Path.cwd())
except ImportError:
    print('Không phải Colab; giữ working directory:', Path.cwd())


Mounted at /content/drive
Working directory: /content/drive/MyDrive/LSTM.MI


In [2]:
import ipaddress
import json
import math
import random
import string
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from urllib.parse import urlsplit

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

MAX_LEN = 75
BATCH_SIZE = 64
NUM_EPOCHS = 17
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 4
THRESHOLD = 0.5
NUM_WORKERS = 2 if device.type == 'cuda' else 0

ALPHABET = string.ascii_lowercase + string.digits + '.-_'
PAD_IDX = 0
UNK_IDX = len(ALPHABET) + 1
char2idx = {char: index + 1 for index, char in enumerate(ALPHABET)}
vocab_size = len(ALPHABET) + 2

OUTPUT_DIR = Path('artifacts/lstm_mi')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / 'best_model_LSTM_MI.pt'

DATA_DIR_CANDIDATES = [Path('data'), Path('../data')]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Không tìm thấy thư mục data. Đã thử: ' +
        ', '.join(str(path.resolve()) for path in DATA_DIR_CANDIDATES)
    )

print('Vocab size:', vocab_size)
print('Data directory:', DATA_DIR.resolve())
print('Checkpoint:', CHECKPOINT_PATH.resolve())


Device: cuda
Vocab size: 41
Data directory: /content/drive/MyDrive/LSTM.MI/data
Checkpoint: /content/drive/MyDrive/LSTM.MI/artifacts/lstm_mi/best_model_LSTM_MI.pt


## Tiền xử lý và kiểm tra dữ liệu

Notebook giữ toàn bộ hostname (cả subdomain/TLD), giống ý tưởng của bản gốc. URL HTTP(S), port,
path, query và dấu chấm cuối được xử lý nhất quán. Unicode được đổi sang Punycode nếu hợp lệ.
Các ký tự ngoài vocabulary dùng `UNK_IDX`.

Việc loại trùng thực hiện trên đúng chuỗi mà model nhìn thấy sau làm sạch và cắt `MAX_LEN`.
Nếu cùng đầu vào nhưng có cả hai nhãn, toàn bộ các hàng đó bị loại. Khi một đầu vào xuất hiện ở
nhiều split, bản ở split sau bị loại để tránh train/validation/test leakage. CSV nguồn không bị sửa.


In [3]:
def clean_domain(value: str) -> str:
    if not isinstance(value, str) or not value.strip():
        raise ValueError('Domain phải là chuỗi không rỗng')

    raw = value.strip()
    parsed = urlsplit(raw if '://' in raw else '//' + raw)
    if parsed.scheme and parsed.scheme.lower() not in ('http', 'https'):
        raise ValueError(f'URL scheme không được hỗ trợ: {parsed.scheme}')
    if parsed.username is not None or parsed.password is not None:
        raise ValueError('Không chấp nhận URL chứa username/password')

    host = (parsed.hostname or '').rstrip('.').lower()
    if not host:
        raise ValueError('Không tìm thấy hostname')
    try:
        ipaddress.ip_address(host)
    except ValueError:
        pass
    else:
        raise ValueError('IP address không phải domain')

    try:
        host = host.encode('idna').decode('ascii')
    except UnicodeError as error:
        raise ValueError('Domain Unicode/IDNA không hợp lệ') from error

    if len(host) > 253:
        raise ValueError('Hostname dài quá 253 ký tự')
    return host


def model_input(value: str) -> str:
    return clean_domain(value)[:MAX_LEN]


def load_frame(path: str, split_name: str) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    required = {'domain_name', 'label'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{split_name}: thiếu cột {sorted(missing)}')
    if frame['domain_name'].isna().any() or frame['label'].isna().any():
        raise ValueError(f'{split_name}: có domain hoặc label bị thiếu')
    if not frame['label'].isin([0, 1]).all():
        bad = frame.loc[~frame['label'].isin([0, 1]), 'label'].unique()[:10]
        raise ValueError(f'{split_name}: label phải là 0/1, thấy {bad}')

    frame = frame.copy()
    frame['source_row'] = np.arange(len(frame))
    cleaned, invalid = [], []
    for row_index, value in enumerate(tqdm(frame['domain_name'], desc=f'Clean {split_name}')):
        try:
            cleaned.append(model_input(value))
        except (TypeError, ValueError) as error:
            cleaned.append(None)
            if len(invalid) < 10:
                invalid.append({'source_row': row_index, 'domain': str(value), 'error': str(error)})
    frame['model_input'] = cleaned
    invalid_count = int(frame['model_input'].isna().sum())
    frame = frame.dropna(subset=['model_input']).copy()
    return frame, {'invalid_rows': invalid_count, 'invalid_examples': invalid}


def prepare_splits(train_path, val_path, test_path):
    paths = {'train': train_path, 'val': val_path, 'test': test_path}
    clean_splits, report, seen = {}, {}, set()

    for name in ('train', 'val', 'test'):
        frame, info = load_frame(paths[name], name)
        input_rows = len(frame) + info['invalid_rows']

        label_counts = frame.groupby('model_input')['label'].nunique()
        conflict_keys = set(label_counts[label_counts > 1].index)
        conflict_rows = int(frame['model_input'].isin(conflict_keys).sum())
        frame = frame[~frame['model_input'].isin(conflict_keys)].copy()

        duplicate_rows = int(frame['model_input'].duplicated().sum())
        frame = frame.drop_duplicates('model_input', keep='first')

        overlap_rows = int(frame['model_input'].isin(seen).sum())
        frame = frame[~frame['model_input'].isin(seen)].copy()
        seen.update(frame['model_input'])
        frame = frame.reset_index(drop=True)

        if set(frame['label'].unique()) != {0, 1}:
            raise ValueError(f'{name}: cần đủ cả hai lớp sau làm sạch')

        clean_splits[name] = frame
        report[name] = {
            'input_rows': input_rows,
            **info,
            'conflicting_label_rows_removed': conflict_rows,
            'duplicate_rows_removed': duplicate_rows,
            'earlier_split_overlap_rows_removed': overlap_rows,
            'output_rows': len(frame),
            'class_counts': {str(k): int(v) for k, v in frame['label'].value_counts().items()},
        }

    return clean_splits, report


splits, data_report = prepare_splits(
    DATA_DIR / 'train_genAI.csv',
    DATA_DIR / 'val_genAI.csv',
    DATA_DIR / 'test_genAI.csv',
)

train_df, val_df, test_df = splits['train'], splits['val'], splits['test']
(OUTPUT_DIR / 'data_cleaning_report.json').write_text(
    json.dumps(data_report, indent=2, ensure_ascii=False), encoding='utf-8'
)

for name, frame in splits.items():
    frame[['source_row', 'domain_name', 'model_input', 'label']].to_csv(
        OUTPUT_DIR / f'{name}_membership.csv', index=False
    )

print(json.dumps(data_report, indent=2, ensure_ascii=False))


Clean train:   0%|          | 0/986316 [00:00<?, ?it/s]

Clean val:   0%|          | 0/123290 [00:00<?, ?it/s]

Clean test:   0%|          | 0/123293 [00:00<?, ?it/s]

{
  "train": {
    "input_rows": 986316,
    "invalid_rows": 0,
    "invalid_examples": [],
    "conflicting_label_rows_removed": 2,
    "duplicate_rows_removed": 0,
    "earlier_split_overlap_rows_removed": 0,
    "output_rows": 986314,
    "class_counts": {
      "0": 730167,
      "1": 256147
    }
  },
  "val": {
    "input_rows": 123290,
    "invalid_rows": 0,
    "invalid_examples": [],
    "conflicting_label_rows_removed": 0,
    "duplicate_rows_removed": 0,
    "earlier_split_overlap_rows_removed": 1,
    "output_rows": 123289,
    "class_counts": {
      "0": 91272,
      "1": 32017
    }
  },
  "test": {
    "input_rows": 123293,
    "invalid_rows": 0,
    "invalid_examples": [],
    "conflicting_label_rows_removed": 0,
    "duplicate_rows_removed": 0,
    "earlier_split_overlap_rows_removed": 2,
    "output_rows": 123291,
    "class_counts": {
      "0": 91272,
      "1": 32019
    }
  }
}


In [4]:
def encode_cleaned_domain(domain: str):
    if not domain:
        raise ValueError('model_input không được rỗng')
    ids = [char2idx.get(char, UNK_IDX) for char in domain]
    length = min(len(ids), MAX_LEN)
    ids = ids[:MAX_LEN] + [PAD_IDX] * (MAX_LEN - length)
    return torch.tensor(ids, dtype=torch.long), length


class DomainDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.domains = frame['model_input'].tolist()
        self.labels = frame['label'].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        tokens, length = encode_cleaned_domain(self.domains[index])
        return tokens, torch.tensor(length, dtype=torch.long), torch.tensor(self.labels[index], dtype=torch.long)


train_ds = DomainDataset(train_df)
val_ds = DomainDataset(val_df)
test_ds = DomainDataset(test_df)

loader_options = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': device.type == 'cuda',
}
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, shuffle=True, generator=train_generator, **loader_options)
val_loader = DataLoader(val_ds, shuffle=False, **loader_options)
test_loader = DataLoader(test_ds, shuffle=False, **loader_options)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')


Train: 986,314 | Val: 123,289 | Test: 123,291


## Model LSTM

`pack_padded_sequence` làm cho LSTM bỏ qua padding. `h_n[-1]` vì vậy là hidden state sau ký tự
cuối thật của mỗi domain, kể cả khi các domain trong cùng batch có độ dài khác nhau.


In [5]:
@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int
    embed_dim: int = 64
    hidden_dim: int = 128
    dropout: float = 0.3
    num_classes: int = 2


class LSTMClassifier(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(config.embed_dim, config.hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(config.dropout)
        self.fc = nn.Sequential(
            nn.Linear(config.hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(config.dropout),
            nn.Linear(128, config.num_classes),
        )

    def forward(self, token_ids, lengths):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed)
        features = self.dropout(hidden[-1])
        return self.fc(features)


MODEL_CONFIG = ModelConfig(vocab_size=vocab_size)
model = LSTMClassifier(MODEL_CONFIG).to(device)

# Smoke check: thêm padding không được thay đổi hidden state vì lengths kiểm soát sequence.
tokens, length = encode_cleaned_domain('google.com')
with torch.inference_mode():
    logits = model(tokens.unsqueeze(0).to(device), torch.tensor([length], device=device))
assert logits.shape == (1, 2)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


LSTMClassifier(
  (embedding): Embedding(41, 64, padding_idx=0)
  (lstm): LSTM(64, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)
Trainable parameters: 118722


## Loss xử lý mất cân bằng

Trọng số được tính trực tiếp theo tần suất lớp. Với dữ liệu hiện tại, `weight[DGA]` phải lớn hơn
`weight[benign]`. Chuẩn hóa mean về 1 chỉ đổi thang loss, không đổi tỷ lệ giữa hai lớp.


In [6]:
def build_class_weights(labels, gamma=0.3):
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    if np.any(counts == 0):
        raise ValueError(f'Mỗi lớp phải có ít nhất một mẫu, counts={counts}')
    weights = np.power(1.0 / counts, gamma)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class_weights = build_class_weights(train_df['label'].to_numpy(), gamma=0.3)
assert class_weights[1] > class_weights[0], 'DGA là lớp thiểu số nên phải có weight lớn hơn'
loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
)

print('Train counts:', np.bincount(train_df['label'].to_numpy(), minlength=2))
print('Class weights [benign, DGA]:', class_weights.tolist())


Train counts: [730167 256147]
Class weights [benign, DGA]: [0.8441522121429443, 1.1558477878570557]


In [7]:
def metrics_from_scores(y_true, y_score, threshold=THRESHOLD):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_score = np.asarray(y_score, dtype=np.float64)
    y_pred = (y_score >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_score)),
        'average_precision': float(average_precision_score(y_true, y_score)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'threshold': float(threshold),
    }, y_pred


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for tokens, lengths, labels in tqdm(loader, desc='train', leave=False):
        tokens = tokens.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(tokens, lengths)
        loss = loss_fn(logits, labels)
        if not torch.isfinite(loss):
            raise RuntimeError('Training loss không hữu hạn')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * len(labels)
        total_samples += len(labels)

    return total_loss / total_samples


@torch.inference_mode()
def evaluate(model, loader, threshold=THRESHOLD):
    model.eval()
    all_labels, all_scores = [], []
    total_loss = 0.0
    total_samples = 0

    for tokens, lengths, labels in tqdm(loader, desc='eval', leave=False):
        tokens = tokens.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(tokens, lengths)
        loss = loss_fn(logits, labels)
        scores = torch.softmax(logits, dim=1)[:, 1]

        total_loss += loss.item() * len(labels)
        total_samples += len(labels)
        all_labels.append(labels.cpu().numpy())
        all_scores.append(scores.cpu().numpy())

    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_scores)
    metrics, y_pred = metrics_from_scores(y_true, y_score, threshold)
    metrics['loss'] = float(total_loss / total_samples)
    return metrics, y_true, y_score, y_pred


## Huấn luyện và chọn checkpoint

Checkpoint được chọn bằng F1 trên validation. Test chưa được đọc trong vòng lặp này.
Early stopping dừng khi validation F1 không cải thiện trong 4 epoch. Thời gian được đo ngay tại
mỗi epoch, nên không cần một vòng train thứ hai.


In [8]:
best_f1 = -math.inf
epochs_without_improvement = 0
history = []
training_started = time.perf_counter()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_started = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader)
    val_metrics, _, _, _ = evaluate(model, val_loader, THRESHOLD)
    epoch_seconds = time.perf_counter() - epoch_started

    row = {
        'epoch': epoch,
        'train_loss': float(train_loss),
        'epoch_seconds': float(epoch_seconds),
        **{f'val_{key}': value for key, value in val_metrics.items()},
        'learning_rate': float(scheduler.get_last_lr()[0]),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'history.csv', index=False)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | acc={val_metrics['accuracy']:.4f} | "
        f"precision={val_metrics['precision']:.4f} | recall={val_metrics['recall']:.4f} | "
        f"f1={val_metrics['f1']:.4f} | auc={val_metrics['roc_auc']:.4f} | "
        f"time={epoch_seconds:.1f}s"
    )

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        epochs_without_improvement = 0
        checkpoint = {
            'format_version': 1,
            'model_name': 'LSTM_MI',
            'epoch': int(epoch),
            'model_state_dict': {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            },
            'model_config': asdict(MODEL_CONFIG),
            'preprocessing': {
                'max_len': int(MAX_LEN),
                'alphabet': ALPHABET,
                'pad_idx': int(PAD_IDX),
                'unk_idx': int(UNK_IDX),
                'char2idx': char2idx,
                'cleaning': 'urlsplit-hostname-idna-lower-v1',
            },
            'label_mapping': {'0': 'Benign', '1': 'DGA'},
            'threshold': float(THRESHOLD),
            'best_validation_f1': float(best_f1),
            'validation_metrics': {
                key: (int(value) if key in {'tn', 'fp', 'fn', 'tp'} else float(value))
                for key, value in val_metrics.items()
            },
            'seed': int(SEED),
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping tại epoch {epoch}; giữ checkpoint có validation F1 tốt nhất.')
        break

total_training_seconds = time.perf_counter() - training_started
print(f'Total training time: {total_training_seconds:.2f} seconds')
print(f'Best validation F1: {best_f1:.4f}')
print('Saved:', CHECKPOINT_PATH.resolve())


train:   0%|          | 0/15412 [00:00<?, ?it/s]

eval:   0%|          | 0/1927 [00:00<?, ?it/s]

Epoch 01/17 | train_loss=0.2438 | val_loss=0.1694 | acc=0.9422 | precision=0.9279 | recall=0.8430 | f1=0.8834 | auc=0.9750 | time=152.1s


train:   0%|          | 0/15412 [00:00<?, ?it/s]

eval:   0%|          | 0/1927 [00:00<?, ?it/s]

...

eval:   0%|          | 0/1927 [00:00<?, ?it/s]

Epoch 15/17 | train_loss=0.0227 | val_loss=0.0715 | acc=0.9817 | precision=0.9694 | recall=0.9599 | f1=0.9646 | auc=0.9961 | time=240.5s


train:   0%|          | 0/15412 [00:00<?, ?it/s]

eval:   0%|          | 0/1927 [00:00<?, ?it/s]

Epoch 16/17 | train_loss=0.0198 | val_loss=0.0768 | acc=0.9814 | precision=0.9694 | recall=0.9587 | f1=0.9640 | auc=0.9960 | time=145.9s
Early stopping tại epoch 16; giữ checkpoint có validation F1 tốt nhất.
Total training time: 2620.43 seconds
Best validation F1: 0.9646
Saved: /content/drive/MyDrive/LSTM.MI/artifacts/lstm_mi/best_model_LSTM_MI.pt


## Nạp checkpoint và đánh giá test đúng một lần

Model được dựng lại từ `model_config` đã lưu. `weights_only=True` tránh lỗi mặc định mới của
PyTorch và không yêu cầu unpickle các đối tượng NumPy. Prediction, classification report và các
metric tổng đều dùng cùng ngưỡng đã lưu trong checkpoint.


In [9]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
if checkpoint.get('format_version') != 1:
    raise ValueError('Checkpoint format không được hỗ trợ')
if checkpoint.get('label_mapping') != {'0': 'Benign', '1': 'DGA'}:
    raise ValueError('Label mapping trong checkpoint không đúng')

loaded_config = ModelConfig(**checkpoint['model_config'])
model = LSTMClassifier(loaded_config)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.to(device).eval()
saved_threshold = float(checkpoint['threshold'])

print(
    f"Loaded epoch {checkpoint['epoch']} | "
    f"validation F1={checkpoint['best_validation_f1']:.4f} | "
    f"threshold={saved_threshold:.3f}"
)

test_metrics, y_test_true, y_test_score, y_test_pred = evaluate(
    model, test_loader, saved_threshold
)
print('\nTEST SET RESULTS')
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
print('\n', classification_report(
    y_test_true, y_test_pred,
    target_names=['Benign', 'DGA'], digits=4, zero_division=0
))

prediction_columns = [
    column for column in ['source_row', 'domain_name', 'model_input', 'subclass']
    if column in test_df.columns
]
test_predictions = test_df[prediction_columns].copy()
test_predictions['truth'] = y_test_true
test_predictions['prediction'] = y_test_pred
test_predictions['score_dga'] = y_test_score
test_predictions['threshold'] = saved_threshold
test_predictions['model'] = 'LSTM_MI'
test_predictions.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

(OUTPUT_DIR / 'test_metrics.json').write_text(
    json.dumps(test_metrics, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Saved test outputs in:', OUTPUT_DIR.resolve())


Loaded epoch 12 | validation F1=0.9646 | threshold=0.500


eval:   0%|          | 0/1927 [00:00<?, ?it/s]


TEST SET RESULTS
{
  "accuracy": 0.9814828332968343,
  "precision": 0.9744702578503958,
  "recall": 0.9536837502732752,
  "f1": 0.9639649593560098,
  "roc_auc": 0.9962530854818755,
  "average_precision": 0.9924384268943423,
  "tn": 90472,
  "fp": 800,
  "fn": 1483,
  "tp": 30536,
  "threshold": 0.5,
  "loss": 0.06546388985959663
}

               precision    recall  f1-score   support

      Benign     0.9839    0.9912    0.9875     91272
         DGA     0.9745    0.9537    0.9640     32019

    accuracy                         0.9815    123291
   macro avg     0.9792    0.9725    0.9758    123291
weighted avg     0.9814    0.9815    0.9814    123291

Saved test outputs in: /content/drive/MyDrive/LSTM.MI/artifacts/lstm_mi


## Dự đoán domain đơn lẻ

`score` là softmax score của lớp DGA. Do model dùng class-weighted loss, score này chưa phải xác
suất đã calibration. Hàm trả schema thuận tiện cho project: `prediction`, `label`, `score`,
`threshold`, `model`.


In [10]:
@torch.inference_mode()
def predict_domain(domain: str, model, threshold=saved_threshold):
    model.eval()
    cleaned = model_input(domain)
    tokens, length = encode_cleaned_domain(cleaned)
    logits = model(
        tokens.unsqueeze(0).to(device),
        torch.tensor([length], dtype=torch.long, device=device),
    )
    score = float(torch.softmax(logits, dim=1)[0, 1].item())
    prediction = int(score >= threshold)
    return {
        'domain': domain,
        'model_input': cleaned,
        'prediction': prediction,
        'label': 'DGA' if prediction else 'Benign',
        'score': score,
        'threshold': float(threshold),
        'model': 'LSTM_MI',
    }


test_domains = [
    'ádadhkjakjdhsk.com',
    'xbvcvloygwnuhiikmuyc.com',
    'tetnolcbxeyenxdeqovp.com',
    'google.co.jp',
    'amazon.it',
    'mozilla.org',
]

for domain in test_domains:
    print(json.dumps(predict_domain(domain, model), ensure_ascii=False))


{"domain": "ádadhkjakjdhsk.com", "model_input": "xn--dadhkjakjdhsk-2db.com", "prediction": 0, "label": "Benign", "score": 2.9488403470168123e-06, "threshold": 0.5, "model": "LSTM_MI"}
{"domain": "xbvcvloygwnuhiikmuyc.com", "model_input": "xbvcvloygwnuhiikmuyc.com", "prediction": 1, "label": "DGA", "score": 0.9994043111801147, "threshold": 0.5, "model": "LSTM_MI"}
{"domain": "tetnolcbxeyenxdeqovp.com", "model_input": "tetnolcbxeyenxdeqovp.com", "prediction": 1, "label": "DGA", "score": 0.9989995360374451, "threshold": 0.5, "model": "LSTM_MI"}
{"domain": "google.co.jp", "model_input": "google.co.jp", "prediction": 0, "label": "Benign", "score": 5.825049811392091e-06, "threshold": 0.5, "model": "LSTM_MI"}
{"domain": "amazon.it", "model_input": "amazon.it", "prediction": 0, "label": "Benign", "score": 0.0035853313747793436, "threshold": 0.5, "model": "LSTM_MI"}
{"domain": "mozilla.org", "model_input": "mozilla.org", "prediction": 0, "label": "Benign", "score": 0.0003303535922896117, "thres

## Ghi chú sử dụng

- Chạy notebook từ trên xuống hoặc **Runtime → Run all**.
- Trên Colab, bật GPU trước khi chạy.
- File dùng để triển khai là `artifacts/lstm_mi/best_model_LSTM_MI.pt`.
- Các CSV membership ghi rõ dòng nào của dữ liệu nguồn thực sự được dùng sau làm sạch.
- Không chạy lại cell train sau khi đã load checkpoint nếu không khởi tạo lại model, optimizer và scheduler.
